In [ ]:
!pip install transformers

In [ ]:
%pip install accelerate

In [ ]:
%pip install pydantic

In [ ]:
import json
import re
from typing import Optional
from pydantic import BaseModel, Field, ValidationError


# Define your expected output schema (customize fields to your task)
class ExtractionSchema(BaseModel):
  date: Optional[str] = Field(
      default=None, description="Extracted date or null"
  )
  location: Optional[str] = Field(
      default=None, description="Extracted location or null"
  )
  summary: Optional[str] = Field(
      default=None, description="Brief summary or null"
  )


def parse_and_validate_output(raw_output: str) -> dict:
  """Safely extracts JSON from raw LLM output and validates it against Pydantic."""
  json_str = None

  # 1. Try direct parsing
  try:
    data = json.loads(raw_output)
    if isinstance(data, dict):
      json_str = data
  except json.JSONDecodeError:
    pass

  # 2. Fallback: Search for JSON block via regex if direct parsing failed
  if not json_str:
    match = re.search(r"(\{.*?\})", raw_output, re.DOTALL)
    if match:
      try:
        data = json.loads(match.group(1))
        if isinstance(data, dict):
          json_str = data
      except json.JSONDecodeError:
        pass

  # 3. If we found a dictionary, validate with Pydantic
  if json_str:
    try:
      validated = ExtractionSchema(**json_str)
      return validated.dict(), True  # Success
    except ValidationError:
      # Schema mismatch, but we got JSON
      return json_str, False

  # 4. Total failure fallback
  return {"date": None, "location": None, "summary": None}, False

# 07 - Metadata extraction with EuroLLM

Run from the `notebooks/` folder. Install once: `%pip install transformers accelerate torch`.

Steps: (1) make a gold-standard template you verify by hand, (2) read the first pages of each PDF, (3) ask EuroLLM for JSON, (4) check the answers against the page text and against your gold rows.

Everything runs locally. Outputs go to `../output/extraction/`.

## 1. Settings

In [ ]:
from pathlib import Path
import json, re, statistics, unicodedata, difflib
import pandas as pd
import pymupdf
from IPython.display import display

PDF_DIR = Path('../pdfs').resolve()
INVENTORY_DIR = Path('../output/inventory').resolve()
OUT_DIR = Path('../output/extraction').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)


MODEL_ID = MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
RUN_SCOPE = 'test'        
FIRST_PAGES = 3              
MAX_INPUT_TOKENS = 3000      
FIELDS = ['title', 'authors', 'doi', 'publication_year', 'document_type', 'language', 'keywords']
LIST_FIELDS = {'authors', 'keywords'}
DOC_TYPES = ['journal_article', 'report', 'thesis', 'conference_paper', 'book_chapter', 'policy_brief', 'other']

read = lambda name: pd.read_csv(INVENTORY_DIR / name, dtype=str, keep_default_na=False, encoding='utf-8-sig')
documents, observations = read('documents.csv'), read('metadata_observations.csv')
print(len(documents), 'documents')

## 2. Gold-standard template

Creates `gold_metadata.csv` once (never overwrites your edits): 40 documents, 25 `dev` and 15 `test`, pre-filled with the PDF-embedded candidates. **Verify every value against the PDF or the Publinova record.**

Fill-in rules: `;` between authors/keywords · `publication_year` as 4 digits · `document_type` from `journal_article, report, thesis, conference_paper, book_chapter, policy_brief, other` · `language` is `en`, `nl` or `mixed` · empty = not checked · `NONE` = checked and absent. Don't tune on `test` until the end.

In [ ]:
GOLD_PATH = OUT_DIR / 'gold_metadata.csv'
if GOLD_PATH.exists():
    print('gold_metadata.csv exists; left untouched.')
else:
    pool = documents[documents.text_status.isin(['extractable', 'partial'])]
    pick = pool.sample(n=min(40, len(pool)), random_state=42)
    cand = lambda d, f: ' | '.join(observations[(observations.document_id == d) & (observations.field_name == f)].observed_value.unique())
    gold = pd.DataFrame({
        'document_id': pick.document_id, 'file_name': pick.file_name,
        'split': ['dev'] * min(15, len(pick)) + ['test'] * max(len(pick) - 15, 0),
        'title': [cand(d, 'title') for d in pick.document_id],
        'authors': [cand(d, 'author') for d in pick.document_id],
        'doi': [cand(d, 'doi') for d in pick.document_id],
        'publication_year': '', 'document_type': '', 'language': '',
        'keywords': [cand(d, 'pdf_keywords') for d in pick.document_id]})
    gold.to_csv(GOLD_PATH, index=False, encoding='utf-8-sig')
    print('Created', GOLD_PATH, '- now verify it by hand.')

## 3. Input and prompt

Each line of the first pages is read with its font size. Lines clearly larger than the page's typical text get a `[BIG]` marker, which helps the model find the title. The prompt ends with `{` so the model continues straight into a JSON object.

In [ ]:
def page_text(page):
    lines = []
    for block in page.get_text('dict', sort=True)['blocks']:
        if block.get('type') != 0:
            continue
        for line in block['lines']:
            spans = [s for s in line['spans'] if s['text'].strip()]
            if spans:
                lines.append((max(s['size'] for s in spans), ''.join(s['text'] for s in spans).strip()))
    if not lines:
        return ''
    typical = statistics.median(size for size, _ in lines)
    return '\n'.join(('[BIG] ' if size >= 1.2 * typical else '') + text for size, text in lines)

def first_relevant_pages(pdf_path, min_chars=200, max_pages=FIRST_PAGES):
    with pymupdf.open(PDF_DIR / pdf_path) as pdf:
        texts = []
        total = 0
        for i in range(min(max_pages + 1, len(pdf))):
            t = page_text(pdf[i])
            texts.append(f'--- page {i + 1} ---\n{t}')
            total += len(t.strip())
            if total >= min_chars:
                break
        return '\n\n'.join(texts)

SYSTEM = (
    'You are a bibliographic metadata extractor. '
    'You read the first page of a document (English or Dutch) and return its metadata as JSON. '
    'Use ONLY information printed in the document. If a field is not printed, use null or []. '
    'Lines starting with [BIG] are in a larger font and are usually the title. '
    'Respond with ONE JSON object and NOTHING else. '
    'Do not repeat these instructions. Do not add any text before or after the JSON. '
    'The JSON must have exactly these keys: '
    'title (string or null), '
    'authors (list of strings), '
    'doi (string or null, without "https://doi.org/" prefix), '
    'publication_year (integer or null, from "Published" or the journal issue year), '
    'document_type (one of: ' + ', '.join(DOC_TYPES) + '), '
    'language ("en", "nl", "mixed", or null), '
    'keywords (list of strings). '
    'Example output for a made-up document:\n'
    '{"title": "The Example Study", "authors": ["A. Smith", "B. Jones"], '
    '"doi": "10.1234/example", "publication_year": 2024, '
    '"document_type": "journal_article", "language": "en", '
    '"keywords": ["alpha", "beta"]}'
)

def user_prompt(text):
    return (
        'Extract the metadata from the document below and return it as a single JSON object.\n\n'
        'DOCUMENT:\n'
        f'{text}'
    )

demo = documents[documents.text_status != 'no_text'].iloc[0]
print(demo.file_name); print(first_relevant_pages(demo.pdf_path)[:1500])